In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [2]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [3]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}

In [4]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [6]:
from toyaikit import tools

agent_tools=tools.Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry,add_entry_tool)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [7]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client=OpenAIClient(
    model='gpt-5.4-mini',
    client=OpenAI()
)


In [8]:
agent=OpenAIResponsesRunner(tools=agent_tools,developer_prompt=instructions,llm_client=llm_client)


In [9]:
result=agent.loop('how can i create a dashbrd in evidentlyai?')
print (result.last_message
       )
result.cost.total_cost

Yes — in Evidently, you create a Dashboard by first having a **Project** with at least one saved **Report**, then adding **Panels** to the Dashboard.

### Main ways to create it

**1) In the UI**
- Go to the Dashboard
- Enter **Edit** mode
- Click **Add Tab** if you want organization into tabs
- Click **Add Panel**
- Configure the panel
- Click **Save** and choose the Tab

This UI flow is available in **Evidently Cloud** and **Enterprise**.

**2) With Python API**
You can create and manage dashboard tabs/panels as code. The docs show examples like:

```python
project.dashboard.add_tab("Another Tab")
```

and adding a panel like:

```python
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Dashboard title",
        size="full",
        values=[],
        plot_params={"plot_type": "text"},
    ),
    tab="My new tab",
)
```

### Important prerequisites
- You must **connect to Evidently Cloud** and **create a Project**
- To populate the Dashboard, you must **save at leas

Decimal('0.02807025')

In [10]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

chat_interface= IPythonChatInterface()
runner_callback=DisplayingRunnerCallback(chat_interface=chat_interface)

result = agent.loop(
    'how can i create evidently dashbrd',
    callback=runner_callback
)


-> Response received


-> Response received


-> Response received


In [11]:
result2= agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)


-> Response received


In [12]:
#QA loop
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [13]:
result=agent.run()

-> Response received


-> Response received


-> Response received


Chat ended.
